# JIT compilation + Tracing + XLA

JIT = Just-In-Time compilation.

Python → execute line by line

JAX does:

Python → trace → build computation graph → compile → run optimized version

In [1]:
import jax
import jax.numpy as jnp

## ✅ Exercise 31 — JIT Compile cube

In [2]:
def cube(x):
    return x ** 3

In [3]:
cube_jit = jax.jit(cube) # JAX does not compile immediately.

## ✅ Exercise 32 — First Run (With Overhead)

In [4]:
cube_jit(10.0) # It waits until you call it. # Lazy compilation.

Array(1000., dtype=float32, weak_type=True)

The first time:

JAX traces the function.

Builds computation graph.

Compiles with XLA.

Executes compiled code.

This takes longer.

## ✅ Exercise 33 — Second Run (Fast)

Now:

No tracing

No compiling

Just runs compiled machine code

In [5]:
cube_jit(10.24)

Array(1073.7418, dtype=float32, weak_type=True)

🧠 Why Faster?

Because Python interpreter is bypassed.

Compiled code runs directly on device.

That’s the power of JAX.

## ✅ Exercise 34 — Store Result

In [6]:
cube_value = cube_jit(10.24)
cube_value

Array(1073.7418, dtype=float32, weak_type=True)

## ✅ Exercise 35 — Disable JIT

In [7]:
# Pure Python execution
# No compilation

with jax.disable_jit():
    cube_value_nojit = cube_jit(10.24)

cube_value_nojit

1073.7418240000002

## ✅ Exercise 36 — eval_shape

Output shape without executing computation.

In [8]:
cube_shape = jax.eval_shape(cube_jit, 10.24)
cube_shape # Scalar output

ShapeDtypeStruct(shape=(), dtype=float32, weak_type=True)

In [9]:
jax.eval_shape(lambda x: x**2, jnp.ones((1000, 1000)))

ShapeDtypeStruct(shape=(1000, 1000), dtype=float32)

## ✅ Exercise 37 — Make JAXPR

This is low-level representation.

JAX intermediate language.

In [10]:
cube_jaxpr = jax.make_jaxpr(cube_jit)(10.24)
cube_jaxpr

{ lambda ; a:f32[]. let
    b:f32[] = jit[
      name=cube
      jaxpr={ lambda ; a:f32[]. let b:f32[] = integer_pow[y=3] a in (b,) }
    ] a
  in (b,) }

## ✅ Exercise 38 — XLA HLO

In [11]:
# Lower and get HLO
xla_ir = cube_jit.lower(10.24).compiler_ir()
print(xla_ir)

module @jit_cube attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  func.func public @main(%arg0: tensor<f32>) -> (tensor<f32> {jax.result_info = "result"}) {
    %0 = stablehlo.multiply %arg0, %arg0 : tensor<f32>
    %1 = stablehlo.multiply %0, %arg0 : tensor<f32>
    return %1 : tensor<f32>
  }
}



JAX traces your function and converts it into pure operations.

Each operation (multiply) is typed and explicit.

This IR (Intermediate Representation) is what XLA compiler sends to GPU/TPU/CPU.

JIT compilation allows the accelerator to fuse operations, optimize memory, and parallelize if possible.

## ✅ Exercise 39 — Benchmark JIT vs Non-JIT

In [12]:
import time

def benchmark(fn, x, runs=100):
    start = time.perf_counter()
    for _ in range(runs):
        jax.block_until_ready(fn(x))
    return (time.perf_counter() - start) / runs

x = jnp.array(10.24)

# Warm up JIT cache
cube_jit(x)

jit_time    = benchmark(cube_jit, x)
nojit_time  = benchmark(cube, x)

print(f"JIT    : {jit_time * 1e6:.2f} µs")
print(f"No JIT : {nojit_time * 1e6:.2f} µs")
print(f"Speedup: {nojit_time / jit_time:.2f}x")
# jax.block_until_ready forces async dispatch to complete.
# Without it, JAX returns a future — timing would be misleading.

JIT    : 7.14 µs
No JIT : 177.89 µs
Speedup: 24.91x


## ✅ Exercise 40 — JIT with static_argnames

In [15]:
import jax.numpy as jnp
from jax import jit

@jit
def power(x, exp):
    return jnp.ones(exp) * x  # shape depends on exp

try:
    power(10.24, exp=3)  # works because 3 is Python int
    power(10.24, exp=jnp.array(3))  # ❌ fails without static_argnames
except Exception  as TypeError:
    print("Error occurred, mark exp as static")
    import functools
    import jax
    import jax.numpy as jnp
    
    @functools.partial(jax.jit, static_argnames=("exp",))
    def power_static(x, exp):
        return jnp.ones(exp) * x

    print(power_static(10.24, exp=3))   # ✅ works

Error occurred, mark exp as static
[10.24 10.24 10.24]
